
<style>
.ormedian-callout {
  border-left: 6px solid #24ABCF;
  background: #F6F8FA;
  padding: 14px 18px;
  margin: 12px 0;
  border-radius: 4px;
  color: #0B1F3B;
}
.ormedian-warning {
  border-left: 6px solid #1281B2;
  background: #EEF5F8;
  padding: 14px 18px;
  margin: 12px 0;
  border-radius: 4px;
  color: #0B1F3B;
}
.ormedian-checkpoint {
  border: 1px solid #B9DCE8;
  background: #FFFFFF;
  padding: 14px 18px;
  margin: 12px 0;
  border-radius: 8px;
  color: #0B1F3B;
}
</style>

<p align="center">
  <img src="../assets/ormedian_session1_banner.png" alt="Ormedian AI Engineering Fundamentals Session 1" width="100%" />
</p>

# Week 1 independent assignment

Complete this notebook during the week. It must run from a fresh kernel using **Run All** and it must tell a coherent story: problem, data, baseline, model, evaluation, errors, experiment and conclusion.



## Submission contract

- Write explanations in your own words.
- Do not use the test set to choose settings.
- Change one main experimental variable.
- Record wrong predictions, not just scores.
- Do not submit code you cannot explain.
- Use relative paths only.



# 0. Setup

Run this complete setup cell. Add no hidden manual steps.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    raise RuntimeError("Start Jupyter from the repository folder.")

sys.path.insert(0, str(ROOT))

from src.data import dataset_summary, get_splits, load_dataset, validate_dataset
from src.evaluation import classification_metrics, error_frame
from src.modelling import build_majority_baseline, build_text_model

DATA_PATH = ROOT / "data" / "processed" / "support_intents.csv"
RANDOM_SEED = 42
print("Ready. Repository root:", ROOT)


# 1. Problem statement

Write one paragraph explaining the imagined support problem.

Then complete:

| Item | Your answer |
|---|---|
| Input |A customer text message|
| Output |The correct purpose of the message |
| User/payer |Support agents and the operations team |
| Action supported |Automatically decide the purpose of each message so it can be routed to the right team (billing, technical support, refunds, etc.)|
| Costly false route | Examples of what happens when it gets it wrong like wastes the agent's time and frustrates the customer |
| Main evaluation metric |Validation macro F1 |
| Out of scope |things the system won't handle |


let's assume e want a customer support system that receives text messages from customers. The system needs to automatically understand the purpose of each message and send it to the right team like billing, technical support, or refunds. If it sends the message to the wrong team, it wastes the agent's time and frustrates the customer. To do this, we will train a text classifier on past labeled messages and test how well it can predict the correct intent for new messages.



In [ ]:
# No model code in this cell.
# Optionally store your framing as a dictionary and display it.
problem = {
    "input": "A customer support message written in plain English",
    "output": "One of the support intent labels",
    "user": "Support agents and customer service operations",
    "costly_mistake": "Routing a refund request to the wrong queue or missing a technical issue",
    "metric": "Validation macro F1, with accuracy as a secondary check",
}
problem


# 2. Load and understand the data

Load the processed data, validate it and show:

- Shape.
- Column names.
- First five rows.
- Number of labels.
- Class distribution.
- Split distribution.
- Blank text count.
- Duplicate text-label count.


In [ ]:
data = load_dataset(DATA_PATH)
validate_dataset(data)

print("Shape:", data.shape)
print("Columns:", list(data.columns))
print("\nFirst five rows:")
display(data.head())
print("\nNumber of labels:", data["label"].nunique())

In [ ]:
summary = dataset_summary(data)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
(data["label"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#24ABCF"))
ax.set_title("Class distribution")
ax.set_xlabel("Label")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## Data interpretation

Answer:

1. What is one example?
2. What is the feature?
3. What is the target?
4. Which columns are metadata?
5. Which columns would create leakage if passed to the model?
6. What important real-world language is not represented by this synthetic dataset?



# 3. Separate train, validation and test

Create the three frames and verify there is no `example_id` overlap.


In [ ]:
train, validation, test = get_splits(data)

print("Train size:", len(train))
print("Validation size:", len(validation))
print("Test size:", len(test))

train_ids = set(train["example_id"])
validation_ids = set(validation["example_id"])
test_ids = set(test["example_id"])

print("No overlap between splits:", not (train_ids & validation_ids) and not (train_ids & test_ids) and not (validation_ids & test_ids))


Explain why:

- Training performance is not enough.
- Validation may guide experiments.
- Test should be opened only after a model choice.



# 4. Majority baseline

Fit the majority baseline and evaluate it on validation data.


In [ ]:
baseline = build_majority_baseline()
baseline.fit(train[["text"]], train["label"])

baseline_pred = baseline.predict(validation["text"])
baseline_metrics = classification_metrics(validation["label"], baseline_pred)

print("Baseline accuracy:", round(baseline_metrics["accuracy"], 4))
print("Baseline macro F1:", round(baseline_metrics["macro_f1"], 4))
print("Baseline weighted F1:", round(baseline_metrics["weighted_f1"], 4))
print("\nClassification report:\n", baseline_metrics["classification_report"])


Interpret the baseline. What would it mean if the real model failed to beat it?



# 5. Main model: TF-IDF unigrams plus logistic regression


In [ ]:
unigram_model = build_text_model(ngram_range=(1, 1))
unigram_model.fit(train["text"], train["label"])

unigram_pred = unigram_model.predict(validation["text"])


In [ ]:
unigram_metrics = classification_metrics(validation["label"], unigram_pred)

print("Accuracy:", round(unigram_metrics["accuracy"], 4))
print("Macro F1:", round(unigram_metrics["macro_f1"], 4))
print("Weighted F1:", round(unigram_metrics["weighted_f1"], 4))
print("\nClassification report:\n")
print(classification_report(validation["label"], unigram_pred, zero_division=0))

In [ ]:
labels = sorted(data["label"].unique())
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    validation["label"],
    unigram_pred,
    labels=labels,
    xticks_rotation=45,
    values_format="d",
    ax=ax,
    colorbar=False,
)
ax.set_title("Validation confusion matrix")
plt.tight_layout()
plt.show()


## Metric interpretation

Choose two labels. For each, explain precision and recall in the context of support routing.



# 6. Error analysis

Create a table of wrong validation predictions and analyse at least three.


In [ ]:
errors = error_frame(validation["text"], validation["label"], unigram_pred)
errors.head(20)


| Text | True | Predicted | Hypothesised cause | Data or model improvement |
|---|---|---|---|---|
| | | | | |
| | | | | |
| | | | | |



# 7. Controlled experiment

Recommended change: `ngram_range=(1, 2)`.

Before running:

- **Question:**
- **Hypothesis:**
- **Variable changed:**
- **Variables held fixed:**
- **Success criterion:**


In [ ]:
bigram_model = build_text_model(ngram_range=(1, 2))
bigram_model.fit(train["text"], train["label"])

bigram_pred = bigram_model.predict(validation["text"])
bigram_metrics = classification_metrics(validation["label"], bigram_pred)

print("Bigram accuracy:", round(bigram_metrics["accuracy"], 4))
print("Bigram macro F1:", round(bigram_metrics["macro_f1"], 4))
print("Bigram weighted F1:", round(bigram_metrics["weighted_f1"], 4))

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "model": "baseline",
            "accuracy": baseline_metrics["accuracy"],
            "macro_f1": baseline_metrics["macro_f1"],
            "weighted_f1": baseline_metrics["weighted_f1"],
        },
        {
            "model": "unigram",
            "accuracy": unigram_metrics["accuracy"],
            "macro_f1": unigram_metrics["macro_f1"],
            "weighted_f1": unigram_metrics["weighted_f1"],
        },
        {
            "model": "bigram",
            "accuracy": bigram_metrics["accuracy"],
            "macro_f1": bigram_metrics["macro_f1"],
            "weighted_f1": bigram_metrics["weighted_f1"],
        },
    ]
)
comparison


## Experiment interpretation

- What changed numerically?
- Did the hypothesis hold?
- Which errors disappeared or appeared?
- Is the difference large enough to matter on this small validation set?
- What would you repeat on a larger dataset?



# 8. Select and evaluate once on test

Use validation macro F1 to select the configuration. Then calculate final test metrics and errors.


In [ ]:
selected_model_name = "bigram" if bigram_metrics["macro_f1"] >= unigram_metrics["macro_f1"] else "unigram"
selected_model = bigram_model if selected_model_name == "bigram" else unigram_model

selected_pred = selected_model.predict(test["text"])
selected_metrics = classification_metrics(test["label"], selected_pred)

print("Selected model:", selected_model_name)
print("Test accuracy:", round(selected_metrics["accuracy"], 4))
print("Test macro F1:", round(selected_metrics["macro_f1"], 4))
print("Test weighted F1:", round(selected_metrics["weighted_f1"], 4))
print("\nTest classification report:\n")
print(classification_report(test["label"], selected_pred, zero_division=0))

In [ ]:
labels = sorted(data["label"].unique())
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    test["label"],
    selected_pred,
    labels=labels,
    xticks_rotation=45,
    values_format="d",
    ax=ax,
    colorbar=False,
)
ax.set_title("Test confusion matrix")
plt.tight_layout()
plt.show()

print("\nTest errors:")
display(error_frame(test["text"], test["label"], selected_pred).head(20))


# 9. Final conclusion

Write 200-400 words covering:

1. The problem and data.
2. The baseline.
3. The chosen model.
4. Validation and test evidence.
5. Most important failure pattern.
6. Dataset limitations.
7. One next data improvement.
8. One next modelling experiment.
9. Why the result is not production-ready.



# 10. Self-assessment

- [ ] Notebook runs from a fresh kernel.
- [ ] All major sections contain Markdown explanation.
- [ ] I can explain every code cell.
- [ ] I did not tune on the test set.
- [ ] I inspected at least three errors.
- [ ] I changed one main variable.
- [ ] I recorded a conclusion supported by evidence.
- [ ] I completed the reflection questions and paper notes.
